# Task 2 - Web Scraping

This notebook replicates the solution from Task 1.

The program:
- asks the user for a Medium article URL
- downloads the article
- extracts the text
- saves the extracted text into a `.txt` file

In [2]:
import os
import sys
import requests
import re
from html import escape
from bs4 import BeautifulSoup

In [3]:
def get_page():
    global url

    url = input("Enter url of a medium article: ")

    # Check if the URL is a Medium article
    if not re.match(r'https?://medium.com/', url):
        print(
            "Please enter a valid website, "
            "or make sure it is a medium article"
        )
        sys.exit(1)

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    res = requests.get(url, headers=headers, timeout=30)

    # Handle 403 Forbidden error
    if res.status_code == 403:
        reader_url = (
            "https://r.jina.ai/http://"
            + url.removeprefix("https://").removeprefix("http://")
        )

        reader_res = requests.get(
            reader_url,
            headers=headers,
            timeout=30
        )

        reader_res.raise_for_status()

        paragraphs = [
            line.strip()
            for line in reader_res.text.splitlines()
            if line.strip()
        ]

        html = "".join(
            f"<p>{escape(paragraph)}</p>"
            for paragraph in paragraphs
        )

        return BeautifulSoup(html, "html.parser")

    res.raise_for_status()

    soup = BeautifulSoup(
        res.text,
        "html.parser"
    )

    return soup

In [4]:
def clean(text):
    rep = {
        "<br>": "\n",
        "<br/>": "\n",
        "<li>": "\n"
    }

    rep = dict(
        (re.escape(k), v)
        for k, v in rep.items()
    )

    pattern = re.compile(
        "|".join(rep.keys())
    )

    text = pattern.sub(
        lambda m: rep[re.escape(m.group(0))],
        text
    )

    text = re.sub(
        r'\<(.*?)\>',
        '',
        text
    )

    return text

In [5]:
def collect_text(soup):
    text = f'url: {url}\n\n'

    para_text = soup.find_all('p')

    print(
        f"paragraphs text = \n{para_text}"
    )

    for para in para_text:
        text += f"{para.text}\n\n"

    return text

In [6]:
def save_file(text):

    if not os.path.exists('./scraped_articles'):
        os.mkdir('./scraped_articles')

    name = url.split("/")[-1]

    print(name)

    fname = f'scraped_articles/{name}.txt'

    with open(
        fname,
        'w',
        encoding='utf-8'
    ) as f:
        f.write(text)

    print(
        f'File saved in directory {fname}'
    )

In [7]:
text = collect_text(get_page())
save_file(text)

paragraphs text = 
[<p>Title: Papa! What is a Neural Network?</p>, <p>URL Source: http://medium.com/@subashgandyer/papa-what-is-a-neural-network-c5e5cc427c7</p>, <p>Published Time: 2018-03-15T12:37:59Z</p>, <p>Markdown Content:</p>, <p>[![Image 1: Subash Gandyer](https://miro.medium.com/v2/resize:fill:32:32/1*RGOBQuxExXL90Z5BmPbuuQ.jpeg)](https://medium.com/@subashgandyer?source=post_page---byline--c5e5cc427c7-----------------------------------------)</p>, <p>10 min read</p>, <p>Mar 15, 2018</p>, <p>It was a cozy Sunday afternoon in the month of February 2018. I just finished my huge customary Sunday lunch spread with family and resting along. Everyone in the family was taking a quick nap for a pre-planned evening outing. Well not everyone, actually.</p>, <p>Press enter or click to view image in full size</p>, <p>![Image 2](https://miro.medium.com/v2/resize:fit:700/1*yqE4wgBedD8ZCWHx3F-V2w.jpeg)</p>, <p>Pictures of Tanishi and Me</p>, <p>My 4-year-old angel came running to me, asked me